In [68]:
import pandas as pd
import warnings
import json
warnings.filterwarnings("ignore")
import numpy as np

In [69]:
data = pd.read_csv('../data/simpsons_script_lines.csv')

In [70]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,true,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,true,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,true,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,true,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


# 1. Filtering Speaking Lines

we detect that speaking line should be a boolean, but it's not as we have lower case intances, correct the dtype 

In [71]:
data['speaking_line'] = data['speaking_line'].replace({'false': False, 'true': True}).astype(bool)

we detect that instances with speaking line = false refer to two narration, sounds of objects of the show (no character associated) or onomatopeias from the characters.
This instances won't hlp us analyzing the words of the characters and therefore we decide to delete them from the dataset to reduce it's size and keep only relevenat instances.
Since the column will be a constant after this filter, we delete it too.


In [72]:
data = data[data['speaking_line']]
data.drop(columns=['speaking_line'], inplace=True)

# 2 Incorrect Word Counts

we notice thah there are 10 incorrect rows, where the spoken words have a different format, the normalized_text is not a sequence of word, but an integer abd the word_count is true instead of an integer.
Most of this lines belong to non-main characters ot groups like 'Entire Town', singers or others.
There are a few of them that belong to the singer tonny bennet. Even though the character exists these and appears to say :Hey, good to see you these lines belong to the background song of the epsiode(https://www.youtube.com/watch?v=dZsOHz1z7Pc). Since tony bennet only says one line and the other ones are not conversation lines, we decide to rease all instances of tonny bennet.
We delte the instances from 'entire town', singers and Revue Cast Members, too, since they don't belong to individual characters and have very little represntation on the entirity of the dataset.


Moe Szyslak: "The reason I left you is simple:

In [73]:
data[data['word_count'] == 'true']

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
8082,17667,59,87,"Singers: (SINGING) ""IT'S THE FIRST ANNUAL MONT...",442000,276.0,636.0,Singers,Springfield Civic Center,IT'S THE FIRST ANNUAL MONTGOMERY BURNS/ AWARD ...,455000,true
71799,81724,282,279,"Revue Cast Members: ""WE'RE THE PERFORMERS YOU ...",1068000,3510.0,2370.0,Revue Cast Members,Branson Theater,WE'RE THE PERFORMERS YOU THOUGHT WERE DEAD / L...,1075000,true
81136,91095,315,284,"Homer Simpson: ""I-M-O--",1220000,2.0,5.0,Homer Simpson,Simpson Home,"""I-M-O--,i-m-o--,1\n91096,315,285,Homer Simpso...",1220000,true
115436,125627,445,216,"Moe Szyslak: ""The reason I left you is simple:",1095000,17.0,15.0,Moe Szyslak,Moe's Tavern,"""The reason I left you is simple:,the reason i...",1095000,true
153015,4260,14,252,"Entire Town: ""SLEIGH BELLS RING / ARE YOU LIST...",1121000,241,211.0,Entire Town,NEIGHBORHOOD,"""SLEIGH BELLS RING / ARE YOU LISTENIN'/,sleigh...",1134000,true
154078,5345,18,199,"Tony Bennett: ""THERE'S A SWINGIN' TOWN I KNOW /",993000,297,239.0,Tony Bennett,Capital City,"THERE'S A SWINGIN' TOWN I KNOW /,theres a swin...",997000,true
154080,5348,18,202,"Tony Bennett: ""PEOPLE STOP AND SCREAM HELLO /",1001000,297,239.0,Tony Bennett,Capital City,"PEOPLE STOP AND SCREAM HELLO /,people stop and...",1004000,true
154082,5351,18,205,"Tony Bennett: ""IT'S THE KIND OF PLACE THAT MAKES",1009000,297,239.0,Tony Bennett,Capital City,"IT'S THE KIND OF PLACE THAT MAKES,its the kind...",1009000,true
154084,5354,18,208,"Tony Bennett: ""AND IT MAKES A KING FEEL LIKE/",1016000,297,239.0,Tony Bennett,Capital City,"AND IT MAKES A KING FEEL LIKE/,and it makes a ...",1019000,true
156870,8152,28,13,"Gulliver Dark: (SINGING) ""THE RULES THAT CONST...",115000,187,332.0,Gulliver Dark,Aztec Theater,THE RULES THAT CONSTRAIN OTHER MEN / MEAN NOTH...,125000,true


In [74]:
data = data[~data['raw_character_text'].isin(['Tony Bennett','Singers','Revue Cast Members','Entire Town'])]

In [75]:
data[data['word_count'] == 'true']

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
81136,91095,315,284,"Homer Simpson: ""I-M-O--",1220000,2.0,5.0,Homer Simpson,Simpson Home,"""I-M-O--,i-m-o--,1\n91096,315,285,Homer Simpso...",1220000,true
115436,125627,445,216,"Moe Szyslak: ""The reason I left you is simple:",1095000,17.0,15.0,Moe Szyslak,Moe's Tavern,"""The reason I left you is simple:,the reason i...",1095000,true
156870,8152,28,13,"Gulliver Dark: (SINGING) ""THE RULES THAT CONST...",115000,187,332.0,Gulliver Dark,Aztec Theater,THE RULES THAT CONSTRAIN OTHER MEN / MEAN NOTH...,125000,true


This error comes from the transcription of the sentcne I am okay as initials I.M.O.K (Disney's subtitles also show the same)
This is actually the end of the previous sentence and not a new one. We will delete this sentence and add the correct normalized ending to the previous one and delete the incorrect one.

Also the sentence Homer says afterwards is lost. He says_  'get it? I am okay'. We will assing this sentence to the current incorrect one

In [76]:
current_sentence = 'last weeks garbage i missed the pick-up date but it doesnt matter because my mom is alive see'
missing_part = 'i am okay'
correct_sentence = current_sentence + ' ' + missing_part
word_count = len(correct_sentence.split())
data.loc[data["id"] == 91094, "word_count"] = word_count
data.loc[data["id"] == 91094, "normalized_text"] = correct_sentence

lost_sentence = 'get it i am okay'
word_count = len(lost_sentence.split())
data.loc[data["id"] == 91095, "word_count"] = word_count
data.loc[data["id"] == 91095, "normalized_text"] = lost_sentence

Moe's sentence error is also fixable, there is a missing part. 
The correct sentence is :  "The reason I left you is simple: I'm gay
We'll correct it on the data

In [77]:
correct_sentence = 'the reason i left you is simple im gay'
word_count = len(correct_sentence.split())
data.loc[data["id"] == 125627, "word_count"] = word_count
data.loc[data["id"] == 125627, "normalized_text"] = correct_sentence

The only one missing is Gulliver Dark's sentence. After checking the epsiode, we have found no appearence of the character. This means that this is most likely a background song the character is singing and not really a line in the show.
We will delete this row 

In [78]:
data = data[data['id'] != 8152]

In [79]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


# 4 Deleting Group characters and character errors

In [80]:
sum(data['character_id'].isna()), sum(data['raw_character_text'].isna())

(2, 3)

In [81]:
data[data['raw_character_text'].isna()]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
86468,96433,334,272,: EVERYWHERE AROUND THE WORLD / THEY'RE COMIN'...,1259000,NaN,681.0,NaN,Ship,EVERYWHERE AROUND THE WORLD / THEY'RE COMIN' T...,everywhere around the world theyre comin to am...,28
142024,152461,546,20,"""ABRAHAM LINCOLN: (FURIOUS) Guess what. I also...",Springfield Elementary School,guess what i also play frankenstein,6.0,NaN,NaN,NaN,NaN,NaN
143085,153593,549,223,: Right here,1091000,NaN,503.0,NaN,Grateful Gelding Stables,Right here,right here,2


delete instances with no character

we have tow instances witch character id missmig and 3 iwth raw chacater_text misisng. The one with id and no raw is a senetnce from Abraham Lincoln (we can see in the raw text), byt the id is not created correctly. we'll do it right. We'll delete the other 2 instances

In [82]:
data.loc[[142023,142024,142025]]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
142023,152460,546,19,"""Abraham Lincoln"": Now, fellow countrymen! A h...",100000,857,3.0,Abraham Lincoln,Springfield Elementary School,"Now, fellow countrymen! A house divided agains...",now fellow countrymen a house divided against ...,8
142024,152461,546,20,"""ABRAHAM LINCOLN: (FURIOUS) Guess what. I also...",Springfield Elementary School,guess what i also play frankenstein,6.0,NaN,NaN,NaN,NaN,NaN
142025,152462,546,21,DOLPH: Douglas is gettin' away!,111000,146,3.0,DOLPH,Springfield Elementary School,Douglas is gettin' away!,douglas is gettin away,4


In [83]:
data.loc[data['id'] == 152461, 'character_id'] = 857
data.loc[data['id'] == 152461, 'raw_character_text'] = 'Abraham Lincoln'
sentence = 'guess what i also play frankenstein'
word_count = len(sentence.split())
data.loc[data['id'] == 152461, 'normalized_text'] = sentence
data.loc[data['id'] == 152461, 'word_count'] = word_count

In [84]:
data = data[data['raw_character_text'].notna()]

With the previous processing step we realized that there are instances that don't identify charaters but groups of them. There are other characters that arfe not that relevant or extras and unnamed ones.For the purpose of this assignemnt we want to focus only on the indviduall characters and therefore will delete all other instances, that augment the size of our dataset without providing any value

In [85]:
len(data[data['raw_character_text'] == "Ticket Seller"])

7

Landlady, Conductor, Husband Of Marge's Friend, Sophisticate #1, Sophisticate #2,Sophisticate #3, Sultan, Convict, Customer, Ticket Seller

potser no fa falta is implement ens quedem arbitrariament alguns o top 10-15 amb més frases

In [86]:
#sorted(data['raw_character_text'].unique().tolist())


# Correcting word_count columns

There are 4 sentences where the spoken_words, normalized_text and word_count columns don't make sense, we'll delete this instances to have a consistent format and not lose any meaningfull data impotance.

In [87]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


In [88]:
wrong_indexes = [152968,130608,117571,86744]
data.loc[wrong_indexes][['raw_character_text','spoken_words', 'normalized_text', 'word_count']]


,raw_character_text,spoken_words,normalized_text,word_count
152968,Thomas Jefferson,"""We hold these truths to be self-evident.,we h...",the humiliation,the fact that it wasn't me. I've never felt so...
130608,Homer Simpson,"""Override self-destruct protocol with authoriz...",THEN:) Uh,"just don't ask me to drive you to the airport."""
117571,Marge Simpson,"""The patrollers were too fast for Eliza and Vi...",a little schmear,and presto -- you're part of the under-clown r...
86744,Patty Bouvier,"""Are you a 'Patty' or a 'Selma?' Take our quiz...",blow me down,"I'm a Selma."""


In [89]:
data = data.drop(index=wrong_indexes)


# Better naming and labeling

??mapejar 10 yr old hommer -> hommer??

# 5 Simplifying labels

we can relabel the characters so they occupy less space in future visualizations, while still being able to identify them.
ex Homer Simposn -> Homer
A version of the same character is categorized using different value for column `raw_character_text`. In some cases the identifier `character_id` would  help us mapping them together, but not it does not group all versions into a single id.
We can find an example of this below


In [90]:
characters_data = data.drop_duplicates(subset='raw_character_text')[['character_id','raw_character_text']]
characters_data[characters_data['character_id'].isin([2,918,2100])]


,character_id,raw_character_text
57,2.0,Homer Simpson
10134,918.0,Homer
13593,2.0,Homer's Brain
35375,2100.0,Flashback Homer
79155,2.0,Homer's Thoughts
106766,2.0,Young Homer


In [91]:
characters_data['character_id'].value_counts()

character_id
2.0       4
9.0       4
1.0       4
8.0       3
603.0     2
         ..
2480.0    1
2479.0    1
2478.0    1
2477.0    1
468       1
Name: count, Length: 6248, dtype: int64

That is why we will map these different versions into a single character 
ex '1-Year-Old Bart' -> Bart
yokel boy yokel girl qui son? Els gitanaos?

s16 epsidoe 2 -> 2nd hommer, don't know what to do

In [92]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


In [93]:
data

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...,...,...,...
158266,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2
158267,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10
158268,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1
158269,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6


In [94]:
homer_list = characters_data[characters_data['raw_character_text'].str.contains('Homer',case = False)]['raw_character_text'].tolist()
excluded = ['HOMER, MARGE AND BART', 'HOMERA', "HOMER-ISH WOMAN'S VOICE", "Knockahomer", "Homer Puppet", 'Homer-ish Husband']
homer_list = [x for x in homer_list if x not in excluded]

marge_list = characters_data[characters_data['raw_character_text'].str.contains('Marge',case = False)]['raw_character_text'].tolist()
excluded = ["Husband Of Marge's Friend",'HOMER, MARGE AND BART']
marge_list = [x for x in marge_list if x not in excluded]

bart_list = characters_data[characters_data['raw_character_text'].str.contains('Bart',case = False)]['raw_character_text'].tolist()
excluded = ['BART & LISA','Bartender','One-eyed Bartender','HOMER, MARGE AND BART','St. Bartholomew','BURLY BARTENDER','Bartleby','Hawaiian Bartender','Australian Bartender',
            'Swedish Bartender',"Bart's Class",'MARGE & BART']
bart_list = [x for x in bart_list if x not in excluded]

lisa_list = characters_data[characters_data['raw_character_text'].str.contains('Lisa',case = False)]['raw_character_text'].tolist()
excluded = ['BART & LISA',"Lisa's Cheering Section"]
lisa_list = [x for x in lisa_list if x not in excluded]

mr_brunst_list = characters_data[characters_data['raw_character_text'].str.contains('Burns',case = False)]['raw_character_text'].tolist()
excluded = ["Burns' Grandfather","Burns' Mother", "Burns's Father",'George Burns''MUTANT MEL/SKINNER/MOE/BURNS/WIGGUM','Ken Burns','Ebenezer Burns',"Burns' Wingman",'BURNS LAWYER']

moe_list = characters_data[characters_data['raw_character_text'].str.contains('Moe',case = False)]['raw_character_text'].tolist()
excluded = ['BETTY & MOE','MUTANT MEL/SKINNER/MOE/BURNS/WIGGUM','MOE HOWARD']
moe_list = [x for x in moe_list if x not in excluded]

frank_list = characters_data[characters_data['raw_character_text'] == 'Frank Grimes']['raw_character_text'].tolist()

milhouse_list = characters_data[characters_data['raw_character_text'].str.contains('Milhouse',case = False)]['raw_character_text'].tolist()

krusty_list = characters_data[characters_data['raw_character_text'].str.contains('Krusty',case = False)]['raw_character_text'].tolist()
excluded = ['Krusty Doll','Bedtime Krusty Doll','Krusty Auditioner']
krusty_list = [x for x in krusty_list if x not in excluded]

ralph_list = characters_data[characters_data['raw_character_text'].str.contains('Ralph',case = False)]['raw_character_text'].tolist()
excluded = ['Boy Resembling Ralph','Ralph Nader']
ralph_list = [x for x in ralph_list if x not in excluded]

chief_wiggum_list = characters_data[characters_data['raw_character_text'].str.contains('Wiggum',case = False)]['raw_character_text'].tolist()
excluded = ['Ralph Wiggum','Sarah Wiggum','MUTANT MEL/SKINNER/MOE/BURNS/WIGGUM']
chief_wiggum_list = [x for x in chief_wiggum_list if x not in excluded]

flanders_list = characters_data[characters_data['raw_character_text'].str.contains('Flanders',case = False)]['raw_character_text'].tolist()

barney_list = characters_data[characters_data['raw_character_text'].str.contains('Barney',case = False)]['raw_character_text'].tolist()
excluded = ["Barney's Boss",'Barney-Shaped Form']

grampa_list = characters_data[characters_data['raw_character_text'].str.contains('Grampa',case = False)]['raw_character_text'].tolist()
excluded = ['Grampa Van Houten','GREAT GRAMPA','Great Grampa Simpson']
grampa_list = [x for x in grampa_list if x not in excluded]

apu_list = characters_data[characters_data['raw_character_text'].str.contains('Apu',case = False)]['raw_character_text'].tolist()
excluded = ["Apu's Father","Apu's Mother",'RAPUNZEL','Catapult Man']
apu_list = [x for x in apu_list if x not in excluded]

nelson_list = characters_data[characters_data['raw_character_text'].str.contains('Nelson',case = False)]['raw_character_text'].tolist()
excluded = ["Nelson's Parrot",'Willie Nelson']
nelson_list = [x for x in nelson_list if x not in excluded]

willie_list = characters_data[characters_data['raw_character_text'].str.contains('Willie',case = False)]['raw_character_text'].tolist()
excluded = ["Willie Nelson","Willie's Lawyer","Willie's Dad","Brother Willie"]
willie_list = [x for x in willie_list if x not in excluded]

duffman_list = characters_data[characters_data['raw_character_text'].str.contains('Duffman',case = False)]['raw_character_text'].tolist()
excluded = ['DUFFMAN VOICE MILHOUSE']
duffman_list = [x for x in duffman_list if x not in excluded]

hibbert_list = characters_data[characters_data['raw_character_text'].str.contains('Hibbert',case = False)]['raw_character_text'].tolist()
excluded = ['Bernice Hibbert','Hibbert Boy',"Hibbert's Father"]
hibbert_list = [x for x in hibbert_list if x not in excluded]

edna_list = characters_data[characters_data['raw_character_text'].str.contains('Krabappel',case = False)]['raw_character_text'].tolist()

bob_list = characters_data[characters_data['raw_character_text'].str.contains('Sideshow Bob',case = False)]['raw_character_text'].tolist()
excluded = ["Sideshow Bob's Mother","Sideshow Bob's Father"]
bob_list = [x for x in bob_list if x not in excluded]

smithers_list = characters_data[characters_data['raw_character_text'].str.contains('Smithers',case = False)]['raw_character_text'].tolist()
excluded = ['ADVISORS & SMITHERS',"Smithers's Wife"]
smithers_list = [x for x in smithers_list if x not in excluded]

lenny_list = characters_data[characters_data['raw_character_text'].str.contains('Lenny',case = False)]['raw_character_text'].tolist()
excluded = ["Lenny's Wife"]
lenny_list = [x for x in lenny_list if x not in excluded]

carl_list = characters_data[characters_data['raw_character_text'].str.contains('Carl',case = False)]['raw_character_text'].tolist()
excluded = ["Carl's Wife",'Carla','Scarlett','Scarlet Pimpernel','Carlotta','Dr. Carlock',"Carl's Father",'Carl Kasell','MOOG/LEONARD/CARLTON/DUM']
carl_list = [x for x in carl_list if x not in excluded]

brockman_list = characters_data[characters_data['raw_character_text'].str.contains('Brockman',case = False)]['raw_character_text'].tolist()
excluded = ["Brockman's Daughter",'Brittany Brockman']
brockman_list = [x for x in brockman_list if x not in excluded] 

reverent_list = characters_data[characters_data['raw_character_text'].str.contains('Lovejoy',case = False)]['raw_character_text'].tolist()
excluded = ['Helen Lovejoy']
reverent_list = [x for x in reverent_list if x not in excluded]

quimby_list = characters_data[characters_data['raw_character_text'].str.contains('Quimby',case = False)]['raw_character_text'].tolist()
excluded = ["Quimby's Lawyer","Quimby's Aide",'Rose Quimby','MARIA SHRIVER KENNEDY QUIMBY']
quimby_list = [x for x in quimby_list if x not in excluded]

comic_book_guy_list = characters_data[characters_data['raw_character_text'].str.contains('Comic',case = False)]['raw_character_text'].tolist()

chalmers_list = characters_data[characters_data['raw_character_text'].str.contains('Chalmers',case = False)]['raw_character_text'].tolist()
excluded = ['Shauna Chalmers']
chalmers_list = [x for x in chalmers_list if x not in excluded]

selma_list = characters_data[characters_data['raw_character_text'].str.contains('Selma',case = False)]['raw_character_text'].tolist()
patty_list = characters_data[characters_data['raw_character_text'].str.contains('Patty',case = False)]['raw_character_text'].tolist()

frink_list = characters_data[characters_data['raw_character_text'].str.contains('Frink',case = False)]['raw_character_text'].tolist()

martin_list = characters_data[characters_data['raw_character_text'].str.contains('Martin',case = False)]['raw_character_text'].tolist()
excluded = ['Dean Martin','Chris Martin','Martin Scorsese']
martin_list = [x for x in martin_list if x not in excluded]

otto_list = characters_data[characters_data['raw_character_text'] == 'Otto Mann']['raw_character_text'].tolist()

mel_list = characters_data[characters_data['raw_character_text'].str.contains('Sideshow Mel',case = False)]['raw_character_text'].tolist()

tony_list = characters_data[characters_data['raw_character_text'].str.contains('Tony',case = False)]['raw_character_text'].tolist()
excluded = ['Tony Blair','Tony Hawk','TONY BOURDAIN']
tony_list = [x for x in tony_list if x not in excluded]

lou_list = characters_data[characters_data['raw_character_text'].str.contains('Lou',case = False)]['raw_character_text'].tolist()
excluded = ['Fallout Boy','Loud Speaker','Loudspeaker','Maya Angelou',"MAN WITH JEWELER'S LOUPE",'Lou Rawls','David McCullough','Jealousy']
lou_list = [x for x in lou_list if x not in excluded]

jimbo_list = characters_data[characters_data['raw_character_text'].str.contains('Jimbo',case = False)]['raw_character_text'].tolist()

kirk_list = characters_data[characters_data['raw_character_text'] == 'Kirk Van Houten']['raw_character_text'].tolist()

agnes_list = characters_data[characters_data['raw_character_text'].str.contains('Agnes',case = False)]['raw_character_text'].tolist()

cletus_list = characters_data[characters_data['raw_character_text'].str.contains('Cletus',case = False)]['raw_character_text'].tolist()
excluded = ["Cletus' Son"]
cletus_list = [x for x in cletus_list if x not in excluded]

In [95]:
map_clean = {
    'Homer': homer_list,
    'Marge': marge_list,
    'Bart': bart_list,
    'Lisa': lisa_list,
    'Mr. Burns': mr_brunst_list,
    'Moe': moe_list,
    'Frank Grimes': frank_list,
    'Milhouse': milhouse_list,
    'Krusty': krusty_list,
    'Ralph': ralph_list,
    'Chief Wiggum': chief_wiggum_list,
    'Flanders': flanders_list,
    'Barney': barney_list,
    'Grampa': grampa_list,
    'Apu': apu_list,
    'Nelson': nelson_list,
    'Groundskeeper Willie': willie_list,
    'Duffman': duffman_list,
    'Dr. Hibbert': hibbert_list,
    'Edna Krabappel': edna_list,
    'Sideshow Bob': bob_list,
    'Smithers': smithers_list,
    'Lenny': lenny_list,
    'Carl': carl_list,
    'Brockman': brockman_list,
    'Rev. Lovejoy': reverent_list,
    'Mayor Quimby': quimby_list,
    'Comic Book Guy': comic_book_guy_list,
    'Chalmers': chalmers_list,
    'Selma': selma_list,
    'Patty': patty_list,
    'Professor Frink': frink_list,
    'Martin Prince': martin_list,
    'Otto': otto_list,
    'Sideshow Mel': mel_list,
    'Fat Tony': tony_list,
    'Lou': lou_list,
    'Jimbo': jimbo_list,
    'Kirk': kirk_list,
    'Agnes Skinner': agnes_list,
    'Cletus': cletus_list
}

with open('../data/map_clean.json', 'w') as f:
    json.dump(map_clean, f,indent=4)

In [96]:
map_functional = {
    variant: canonical
    for canonical, variants in map_clean.items()
    for variant in variants
}

with open('../data/map_functional.json', 'w') as f:
    json.dump(map_functional, f,indent=4)

In [97]:
data['raw_character_text'] = data['raw_character_text'].map(map_functional)
data = data.dropna(subset=['raw_character_text'])

# Dealin with missing values on the normalized text

there are a few instances that belong to noises, and they don't produce any words. We'll delete them from the dataset

In [98]:
data = data[data['normalized_text'].notna()]

# Dealing with strings on the normalized text

some rows have -- on the normalized words column. This belong to sentences getting interrupted, but the word count column counts it as a word.
ex now kirk its only a game sometimes we --  : https://www.youtube.com/watch?v=DXeRCJzrQEA

We wi'll delete the '-' from the normalized_text and recount the words for them

In [99]:
indexes = data[data['normalized_text'].str.contains('-')].index
data.loc[indexes, 'normalized_text'] = data.loc[indexes, 'normalized_text'].str.replace('-', ' ', regex=False)
data.loc[indexes, 'word_count'] = data.loc[indexes, 'normalized_text'].str.split().apply(len)

In [100]:
data[data['normalized_text'].str.contains('--')]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count


# Dealing with outliers

We find and instance with a suposed word count of 571000. The issue we see is that the spoken words are part of the lyrics of the 'Wannabe' song, while the normalzied text references a sentence that Bart said. The Wanabee song sounds in the begining of the scene of the sentence so it both got mixed up in a single instance. We'll correct it to reflect Bart's sentence. We alos normalize the text since it's not

In [101]:
data.loc[data['id'] == 62483,'raw_text'] = None
data.loc[data['id'] == 62483,'spoken_words'] = None
data.loc[data['id'] == 62483,'raw_character_text'] = 'Bart'
sentence = 'dad he wants you to blow your horn.'    
data.loc[data['id'] == 62483,'normalized_text'] = sentence
word_count = len(sentence.split())
data.loc[data['id'] == 62483,'word_count'] = word_count

Bart also has an outlier 

In [102]:
data[data['id'] == 5432]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
154163,5432,19,5,"Bart Simpson: (WRITING) ""One o'clock -- still ...",104000,8,5.0,Bart,Simpson Home,"""One o'clock -- still just a potato.,one ocloc...",neighbor! The Lord's certainly given us a beau...,108000


In [103]:
sentence = "one o clock still just a potato neighbor the lord is certainly given us a beautiful day today huh?"
wprd_count = len(sentence.split())
data.loc[data['id'] == 5432, 'normalized_text'] = sentence
data.loc[data['id'] == 5432, 'word_count'] = wprd_count

marge outlier

In [104]:
data[data['id'] == 69807]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
59908,69807,243,292,"Marge Simpson: (CONTINUING) ""My gold is in the...",1145000,1.0,422.0,Marge,White House,"""My gold is in the heart of every freedom-lovi...","crap!""",1145000


In [105]:
sentence = 'my gold is in the heart of every freedom loving american'
word_count = len(sentence.split())
data.loc[data['id'] == 69807, 'normalized_text'] = sentence
data.loc[data['id'] == 69807, 'word_count'] = word_count

Homer otulier

In [106]:
data.loc[data['id'] == 88907]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
78951,88907,308,85,"Homer Simpson: (READING) ""Nausea... cravings.....",393000,2.0,15.0,Homer,Moe's Tavern,"""Nausea... cravings... knocked-up feeling... S...",the popular singer slash songwriter slash puzz...,409000


the popular singer slash songwriter slash puzzle piece."

In [107]:
sentence = "nausea cravings knocked up feeling she was pregnant with bart and that is the reason she stayed with me"
word_count = len(sentence.split())
data.loc[data['id'] == 88907, 'normalized_text'] = sentence
data.loc[data['id'] == 88907, 'word_count'] = word_count

Lisa outlier 1

In [108]:
data.loc[data['id'] == 87170]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
77228,87170,302,13,"Lisa Simpson: (READING) ""Molochai desiratum ma...",147000,9.0,5.0,Lisa,Simpson Home,"""Molochai desiratum maledictu... nosferatu asc...","Mad libs!""",147000


In [109]:
sentence = 'molochai desiratum maledictu nosferatu ascendum corporalis'
word_count = len(sentence.split())
data.loc[data['id'] == 87170, 'normalized_text'] = sentence
data.loc[data['id'] == 87170, 'word_count'] = word_count

lisa outlier 2

In [110]:
data.loc[data['id'] == 111237]

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
101152,111237,390,16,"Lisa Simpson: ""...Hitachee Tribe. (NERVOUS GIG...",117000,9.0,6.0,Lisa,KITCHEN,"""...Hitachee Tribe.,hitachee tribe,2\n111238,3...",is it wrong for me to appropriate the culture ...,14


In [111]:
sentence = 'is it wrong for me to appropriate the culture of a long suffering people'
word_count = len(sentence.split())
data.loc[data['id'] == 111237, 'normalized_text'] = sentence
data.loc[data['id'] == 111237, 'word_count'] = word_count

# 3 Irrelevant Columns

there are a few columns that are also irrelevant for our use case.
* timestamp_in_ms -> we don't care about the moment in the episode the character speaks
* location_id -> we don't care about location
* raw_location_text -> same as the one above
* raw_text -> same as normalized text and spoken words, but adds noise such as emotions between brackets, which we are not interested in this assingment and make processing more complicated
* spoken_words -> it's repetitive having it's normalized version 
* character_id -> we have the character
We can delete this columns


In [112]:
data.drop(columns = ['timestamp_in_ms','location_id','raw_location_text','raw_text','spoken_words','character_id'],inplace=True)

# 4 Adding Season and number in season

we don't have a clear column indicating the season the epsiodes belong to, even though they can be deuced from the epsiode_id column, which indicates the epsiode without distingushing between seasons.
Since maaping this epsidoes to their respective seasons would requiere manually checking each one of them, we can use the dataset of the previous assingment that already has a defined 'season' column to create the column in our dataset

Since for the clean version of the dataset of the previous assignment we deleted all epsiodes of the last available season (it only had 4 episdes) to help our interests, we won't use this version for this assingment since we can use that information. We will read the raw one

In [113]:
data_seasons = pd.read_csv('../data/simpsons_episodes_raw.csv')[['season','number_in_season','number_in_series']]

In [114]:
episodes_daat1 = set(data['episode_id'])
episodes_data2 = set(data_seasons['number_in_series'])
episodes_daat1 - episodes_data2

set()

we see that all epsiodes in our script lines dataset are in the datset from the previous quarter

In [115]:
print(episodes_data2 - episodes_daat1)
len(episodes_data2 - episodes_daat1)

{550, 424, 569, 441, 570, 571, 572, 573, 575, 576, 577, 578, 579, 580, 581, 582, 583, 574, 447, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600}


36

However this is not true when inverting the order, the dataset from the previous assignment has 32 more episodes. The second source used on the previous assignment also has this same issue. We explored extra datasets such as this one: https://github.com/jcrodriguez1989/thesimpsons or https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-02-04/readme.md but to no success. No datasets contained the missing epsiodes

In [116]:
data = data.merge(data_seasons, left_on='episode_id', right_on='number_in_series', how='left')
data.drop(columns=['number_in_series'], inplace=True)
cols = data.columns.tolist()
cols.insert(cols.index('episode_id'), cols.pop(cols.index('season')))
cols.insert(cols.index('episode_id'), cols.pop(cols.index('number_in_season')))
data = data[cols]
data

,id,season,number_in_season,episode_id,number,raw_character_text,normalized_text,word_count
0,9550,2,19,32,210,Lisa,wheres mr bergstrom,3
1,9552,2,19,32,212,Lisa,that life is worth living,5
2,9553,2,19,32,213,Edna Krabappel,the polls will be open from now until the end ...,33
3,9554,2,19,32,214,Martin Prince,i dont think theres anything left to say,8
4,9555,2,19,32,215,Edna Krabappel,bart,1
...,...,...,...,...,...,...,...,...
98510,9538,2,19,32,198,Lisa,does bart have to be there,6
98511,9539,2,19,32,199,Marge,yes,1
98512,9540,2,19,32,200,Lisa,can we do it this week,6
98513,9542,2,19,32,202,Lisa,mr bergstrom we request the pleasure of your c...,33


# 5 Renamiming to keep columns undestandable

column number identifies the line within each, season. We find this name rather confusing, we'll change it to one that explains the variable better

In [117]:
data.rename(columns={'number': 'line_number_in_episode','raw_character_text': 'character'}, inplace=True)  

# Adding Images for plottinh later

In [118]:
map_clean.keys()

dict_keys(['Homer', 'Marge', 'Bart', 'Lisa', 'Mr. Burns', 'Moe', 'Frank Grimes', 'Milhouse', 'Krusty', 'Ralph', 'Chief Wiggum', 'Flanders', 'Barney', 'Grampa', 'Apu', 'Nelson', 'Groundskeeper Willie', 'Duffman', 'Dr. Hibbert', 'Edna Krabappel', 'Sideshow Bob', 'Smithers', 'Lenny', 'Carl', 'Brockman', 'Rev. Lovejoy', 'Mayor Quimby', 'Comic Book Guy', 'Chalmers', 'Selma', 'Patty', 'Professor Frink', 'Martin Prince', 'Otto', 'Sideshow Mel', 'Fat Tony', 'Lou', 'Jimbo', 'Kirk', 'Agnes Skinner', 'Cletus'])

In [119]:
images = {
    'Marge': 'https://img.icons8.com/doodle/48/marge-simpson.png',
    'Homer': 'https://img.icons8.com/doodle/48/homer-simpson.png',
    'Bart': 'https://img.icons8.com/doodle/48/bart-simpson.png',
    'Lisa': 'https://img.icons8.com/doodle/48/lisa-simpson.png',
    'Mr. Burns': 'https://img.icons8.com/doodle/48/charles-montgomery-burns.png',
    'Moe': 'https://img.icons8.com/doodle/48/bartender-mo.png',
    'Frank Grimes': 'https://static.simpsonswiki.com/images/a/a0/Tapped_Out_Frank_Grimes_Icon_-_Relaxed.png',
    'Milhouse': 'https://img.icons8.com/doodle/48/milhouse-van-houten.png',
    'Krusty':'https://img.icons8.com/doodle/48/krusty-the-clown.png',
    'Ralph': 'https://img.icons8.com/doodle/48/ralf.png',
    'Chief Wiggum':'https://img.icons8.com/doodle/48/clancy-wiggum.png',
    'Flanders':'https://img.icons8.com/doodle/48/ned-flanders.png',
    'Barney': 'https://img.icons8.com/doodle/48/barney-gumble.png',
    'Grampa': 'https://img.icons8.com/doodle/48/abraham-simpson.png',
    'Apu': 'https://static.simpsonswiki.com/images/0/08/Tapped_Out_Apu_Icon_-_Happy.png',
    'Nelson':'https://static.simpsonswiki.com/images/0/0c/Tapped_Out_Nelson_Icon_-_Brave.png',
    'Groundskeeper Willie':'https://static.simpsonswiki.com/images/4/40/Tapped_Out_Chainsaw_Willie_Icon_-_Happy.png',
    'Duffman':'https://static.simpsonswiki.com/images/1/18/Tapped_Out_Duffman_Icon.png',
    'Dr. Hibbert':'https://static.simpsonswiki.com/images/d/d5/Tapped_Out_Dr._Hibbert_Icon.png',
    'Edna Krabappel':'https://static.simpsonswiki.com/images/1/14/Tapped_Out_Mrs._Krabappel_Icon.png',
    'Sideshow Bob':'https://static.simpsonswiki.com/images/6/68/Tapped_Out_Sideshow_Bob_Icon_-_Serious.png',
    'Smithers':'https://static.simpsonswiki.com/images/3/3a/Tapped_Out_Smithers_Icon.png',
    'Lenny':'https://static.simpsonswiki.com/images/6/6f/Tapped_Out_Lenny_Icon.png',
    'Carl':'https://static.simpsonswiki.com/images/a/a4/Tapped_Out_Carl_Icon.png',
    'Brockman':'https://static.simpsonswiki.com/images/3/35/Tapped_Out_Brockman_Sidebar.png',
    'Rev. Lovejoy':'https://static.simpsonswiki.com/images/4/40/Tapped_Out_Rev._Lovejoy_Sidebar.png',
    'Mayor Quimby':'https://static.simpsonswiki.com/images/e/e3/Tapped_Out_Quimby_Icon.png',
    'Comic Book Guy':'https://static.simpsonswiki.com/images/a/a9/Tapped_Out_Comic_Book_Guy_Icon.png',
    'Chalmers':'https://static.simpsonswiki.com/images/5/58/Tapped_Out_Chalmers_Icon.png',
    'Selma':'https://static.simpsonswiki.com/images/8/8f/Tapped_Out_Selma_Icon.png',
    'Professor Frink':'https://static.simpsonswiki.com/images/0/09/Tapped_Out_Professor_Frink_Icon_-_Deadpan.png',
    'Patty':'https://static.simpsonswiki.com/images/3/31/Tapped_Out_Patty_Sidebar.png',
    'Martin Prince':'https://static.simpsonswiki.com/images/d/da/Tapped_Out_Martin_Icon.png',
    'Otto':'https://static.simpsonswiki.com/images/4/48/Tapped_Out_Conductor_Otto_Sidebar.png',
    'Sideshow Mel':'https://static.simpsonswiki.com/images/6/6d/Tapped_Out_Sideshow_Mel_Icon.png',
    'Fat Tony':'https://static.simpsonswiki.com/images/1/1b/Tapped_Out_Fat_Tony_Icon.png',
    'Lou':'https://static.simpsonswiki.com/images/c/cf/Tapped_Out_Lou_Icon.png',
    'Jimbo':'https://static.simpsonswiki.com/images/f/f3/Tapped_Out_Jimbo_Icon.png',
    'Kirk':'https://static.simpsonswiki.com/images/6/61/Tapped_Out_Acorn_Kirk_Icon.png',
    'Agnes Skinner':'https://static.simpsonswiki.com/images/b/b1/Tapped_Out_Agnes_Sidebar.png',
    'Cletus': 'https://static.simpsonswiki.com/images/a/a0/Tapped_Out_Cletus_Icon_-_Smiling.png'
}
with open('../data/characters_images.json', 'w') as f:
    json.dump(images, f,indent=4)

In [120]:
print(f'unique characters: {data["character"].nunique()}')
data = data[data['character'].isin(list(images.keys()))]
# filter to characters appearing in at least 5 seasons
data = data.groupby('character').filter(lambda x: x['season'].nunique() >= 5)
print(f'unique characters after filtering: {data["character"].nunique()}')
print(f'Excluded characters: {set(map_clean.keys()) - set(data["character"].unique())}')


unique characters: 41
unique characters after filtering: 40
Excluded characters: {'Frank Grimes'}


In [121]:
data

,id,season,number_in_season,episode_id,line_number_in_episode,character,normalized_text,word_count
0,9550,2,19,32,210,Lisa,wheres mr bergstrom,3
1,9552,2,19,32,212,Lisa,that life is worth living,5
2,9553,2,19,32,213,Edna Krabappel,the polls will be open from now until the end ...,33
3,9554,2,19,32,214,Martin Prince,i dont think theres anything left to say,8
4,9555,2,19,32,215,Edna Krabappel,bart,1
...,...,...,...,...,...,...,...,...
98510,9538,2,19,32,198,Lisa,does bart have to be there,6
98511,9539,2,19,32,199,Marge,yes,1
98512,9540,2,19,32,200,Lisa,can we do it this week,6
98513,9542,2,19,32,202,Lisa,mr bergstrom we request the pleasure of your c...,33


In [122]:
data['word_count'] = pd.to_numeric(data['word_count'], errors='coerce').astype(float)

In [123]:
data.to_csv('../data/simpsons_script_lines_cleaned.csv', index=False)

# Data Q2

In [124]:
data_line_plot = data.groupby(['season','character'])['word_count'].sum().reset_index()
data_line_plot['image'] = data_line_plot['character'].map(images)
data_line_plot.to_csv('../data/data_Q2.csv', index=False)

# Data Q1 & Q5

In [125]:
data_word_counts = data.drop(columns=['line_number_in_episode','normalized_text'])
data_word_counts = data_word_counts.groupby(['character','episode_id'])['word_count'].sum().reset_index()
data_word_counts.to_csv('../data/data_Q1-2.csv', index=False)

data_word_counts = data_word_counts.groupby('character')['word_count'].sum().reset_index()
data_word_counts['image'] = data_word_counts['character'].map(images)
data_word_counts.sort_values(by='word_count', ascending=False, inplace=True)
data_word_counts.to_csv('../data/data_Q1-1.csv', index=False)

In [126]:
data_sentence_counts = data.groupby(['season','number_in_season','episode_id','character']).size().reset_index(name='sentence_count')
data_sentence_counts.to_csv('../data/data_Q5-2.csv', index=False)

data_sentence_counts = data_sentence_counts.groupby('character')['sentence_count'].sum().reset_index()
data_sentence_counts['image'] = data_sentence_counts['character'].map(images)
data_sentence_counts.sort_values(by='sentence_count', ascending=False, inplace=True)
data_sentence_counts.to_csv('../data/data_Q5-1.csv', index=False)

# Data Q3 & Q4

In [127]:
data_q3 = data.groupby(['season', 'number_in_season', 'episode_id','character'])['word_count'].sum().reset_index()
data_q3['image'] = data_q3['character'].map(images)
data_q3.to_csv('../data/data_Q3.csv', index=False)


In [129]:
data_q4 = data.drop(columns=['id','normalized_text'])
data_q4['image'] = data_q4['character'].map(images)
data_q4.to_csv('../data/data_Q4.csv', index=False)